In [24]:
from dataclasses import dataclass
from dotenv import load_dotenv
from openai import OpenAI
import os

load_dotenv()

@dataclass(frozen=True)
class Provider:
    """One provider to reliably route requests accross all inference providers """

    name: str
    env_var: str
    is_free: bool
    base_url: str | None
    model: str

PROVIDERS = [
    Provider("OpenRouter", "OPENROUTER_API_KEY", True, "https://openrouter.ai/api/v1", "openai/gpt-4o-mini"),
    Provider("OpenAI", "OPENAI_API_KEY", False, None, "gpt-4o-mini"),
    Provider("Groq", "GROQ_API_KEY", True, "https://api.groq.com/openai/v1", "openai/gpt-oss-120b"),
]


def select_provider() -> Provider:
    for provider in PROVIDERS:
        if os.getenv(provider.env_var):
            return provider

    expected = ", ".join(p.env_var for p in PROVIDERS)
    raise RuntimeError(f"No provider key set, add one of {expected} to your environment variables")

def build_client(provider: Provider) -> OpenAI:

    api_key = os.getenv(provider.env_var)
    if provider.base_url is None:
        return OpenAI(api_key=api_key)

    return OpenAI(
        api_key=api_key,
        base_url=provider.base_url,
    )


def have_any_key() -> bool:
    return any(os.getenv(p.env_var) for p in PROVIDERS)

print("Found a provider key." if have_any_key() else "No provider key found.")

Found a provider key.


In [25]:
def llm_reply(prompt: str) -> str:
    provider = select_provider()
    print(f"Using {provider.name} provider")
    client = build_client(provider)
    result = client.responses.create(
        model=provider.model,
        max_output_tokens=200,
        input=[
            {
                "role": "user",
                "content": prompt,
            }
        ]
    )

    return result.output_text

In [26]:
prompt = "Who was the PM of India before 2020 ? Answer in a single sentence"

try:
    print(llm_reply(prompt))
except Exception as e:
    print(f"Error: {e}")

Using OpenRouter provider
The Prime Minister of India before 2020 was Narendra Modi, who has held the position since May 26, 2014.


When no key is found error from line 30, 31 1st block


Error: No provider key set, add one of OPENAI_API_KEY, GROQ_API_KEY to your environment variables

In [27]:
def chat_reply(messages: list[dict]) -> str:
    provider = select_provider()

    client = build_client(provider)

    result = client.responses.create(
        model=provider.model,
        max_output_tokens=1000,
        input=messages,
    )

    return result.output_text

In [28]:
conversation = []

In [29]:
conversation.append({"role": "user", "content": "What is the capital of India?"})

In [30]:
reply_from_llm = chat_reply(conversation)
print(reply_from_llm)

conversation.append({
    "role": "assistant",
    "content": reply_from_llm,
})


The capital of India is New Delhi.


In [31]:
conversation

[{'role': 'user', 'content': 'What is the capital of India?'},
 {'role': 'assistant', 'content': 'The capital of India is New Delhi.'}]

In [33]:
conversation.append({
    "role": "user",
    "content": "what is their GDP ? ",
})

In [34]:
conversation

[{'role': 'user', 'content': 'What is the capital of India?'},
 {'role': 'assistant', 'content': 'The capital of India is New Delhi.'},
 {'role': 'user', 'content': 'what is their GDP ? '}]

In [35]:
llm_reply = chat_reply(conversation)
print(llm_reply)

As of my last knowledge update in October 2023, India's Gross Domestic Product (GDP) was estimated to be around $3.5 trillion in nominal terms, making it one of the largest economies in the world. However, GDP figures are frequently updated, so it's advisable to check the latest data from reliable financial sources for the most current information.


## Key Takeaways

- A bare **model** is a one-shot function: text in, text out, no memory.
- A **chatbot** = model + a **transcript list** of `{"role", "content"}` dictionaries. Sending that list back each turn *is* memory.
- A real call is still text-in / text-out, just wrapped: a messages list goes in, and comes out (OpenAI dialect).
- **All three providers speak the OpenAI dialect** — Groq and OpenRouter reuse the `openai` SDK with a different `base_url`, so one call path covers them all.